In [1]:
# Data collection
import nltk

# Download Gutenberg corpus
nltk.download('gutenberg')

from nltk.corpus import gutenberg
import pandas as pd

# Load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')

# Save the file
with open('hamlet.txt', 'w', encoding='utf-8') as file:
    file.write(data)

print("File saved successfully!")

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


File saved successfully!


In [2]:
print(gutenberg.fileids())

['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


In [3]:
## Data Preprocessing

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Load the dataset
with open('hamlet.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
print(total_words)

4818


In [4]:
## create the input sequences
input_sequences=[]
for line in text.split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
        n_gram_sequence=token_list[:i+1]
        input_sequences.append(n_gram_sequence)


In [5]:
## pad sequences
max_sequence_len=max([len(x) for x in input_sequences])
input_sequences=np.array(pad_sequences(input_sequences,maxlen=max_sequence_len,padding='pre'))


In [6]:
## create the predictors and label
import tensorflow as tf
x,y=input_sequences[:,:-1],input_sequences[:,-1]
y=tf.keras.utils.to_categorical(y,num_classes=total_words)

In [7]:
 X_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [9]:
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.models import Sequential

# Define the model
model = Sequential()

model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

model.add(GRU(150, return_sequences=True))

model.add(Dropout(0.2))

model.add(GRU(100))

model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
## Train the model
history=model.fit(X_train,y_train,epochs=10,validation_data=(x_test,y_test),verbose=1)


Epoch 1/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 51s 72ms/step - accuracy: 0.0320 - loss: 7.0116 - val_accuracy: 0.0336 - val_loss: 6.7806
Epoch 2/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 74s 60ms/step - accuracy: 0.0414 - loss: 6.4971 - val_accuracy: 0.0468 - val_loss: 6.7868
Epoch 3/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.0521 - loss: 6.2692 - val_accuracy: 0.0616 - val_loss: 6.7433
Epoch 4/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 39s 60ms/step - accuracy: 0.0675 - loss: 6.0219 - val_accuracy: 0.0707 - val_loss: 6.7450
Epoch 5/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.0831 - loss: 5.7593 - val_accuracy: 0.0701 - val_loss: 6.7677
Epoch 6/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 58ms/step - accuracy: 0.0899 - loss: 5.5246 - val_accuracy: 0.0715 - val_loss: 6.8836
Epoch 7/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 59ms/step - accuracy: 0.0990 - loss: 5.3014 - val_accuracy: 0.0696 - val_loss: 6.9542
Epoch 8/10
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 60ms/step - accuracy: 0.1041 - loss: 5.0831 - 

In [ ]:
def predict_next_word(model, tokenizer, input_text, n_words):

    for _ in range(n_words):

        # Convert text into integer tokens
        token_list = tokenizer.texts_to_sequences([input_text])[0]

        # Pad the sequence
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len-1,
            padding='pre'
        )

        # Predict probabilities
        predicted = model.predict(token_list, verbose=0)

        # Get the index of the word with highest probability
        predicted_word_index = np.argmax(predicted, axis=-1)[0]

        # Convert index back to word
        predicted_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                predicted_word = word
                break

        # Add predicted word to input
        input_text += " " + predicted_word

    return input_text

In [ ]:
input_text = "to be or not be"

print(f"Input text: {input_text}")

# Get the required sequence length from the trained model
max_sequence_len = model.input_shape[1] + 1

# Generate 3 new words
predicted_text = predict_next_word(
    model,
    tokenizer,
    input_text,
    3
)

print(f"Predicted text: {predicted_text}")

In [ ]:
## Save the model
model.save("next_word_lstm.h5")
## Save the tokenizer
import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
input_text="To be bad is beter than"
print(f"Input text: {input_text}")
predicted_text=predict_next_word(model,tokenizer,input_text,3)
print(f"Predicted text: {predicted_text}")